# Product Evidence Platform — Azure ML Runner

This is the only supported notebook.

1. Copy `.env.example` to `.env` and replace every placeholder.
2. Place your private feature JSON in `inputs/private/<name>.json`.
3. Run `./scripts/azureml_startup.sh` from the Azure ML terminal.
4. Run the health-check cell, then submit one product or a CSV batch.

Every product uses exactly three searches: requested retailer in country, other retailers in country, and global fallback. A `primary_url` is returned only after browser-openable, exact-product, scrapability, complete requested-feature, and durable non-expiring URL checks pass.

`COMPLETED` and `REVIEW_REQUIRED` are successful terminal workflow states. `REVIEW_REQUIRED` means execution completed but no URL passed every mandatory acceptance gate. Only `FAILED` represents an execution failure.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import time
from pathlib import Path
from pprint import pprint
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root containing docker-compose.yml")

PROJECT_ROOT = find_project_root()
AGENT_URL = os.getenv("PRODUCT_AGENT_URL", "http://127.0.0.1:8788").rstrip("/")
POLL_SECONDS = 3
TERMINAL_STATUSES = {"COMPLETED", "REVIEW_REQUIRED", "FAILED"}

def api_json(method: str, path: str, payload: dict | None = None, timeout: int = 30) -> dict:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(f"{AGENT_URL}{path}", data=body, method=method, headers={"Content-Type": "application/json"})
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Agent API returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(f"Cannot reach {AGENT_URL}. Run ./scripts/azureml_startup.sh first.") from exc

def check_health() -> dict:
    health = api_json("GET", "/health", timeout=15)
    if health.get("status") != "healthy":
        raise RuntimeError(f"Platform is not healthy: {json.dumps(health, indent=2)}")
    configuration = health.get("configuration") or {}
    if not configuration.get("three_stage_contract_enforced"):
        raise RuntimeError("Agent is not running the strict three-stage contract")
    if configuration.get("serpapi_request_limit") != 3:
        raise RuntimeError("Agent search credit limit is not exactly 3")
    return health

def submit_product(product: dict, feature_set: str) -> str:
    response = api_json("POST", "/v1/jobs", {"product": product, "feature_set": feature_set})
    return response["job_id"]

def wait_for_job(job_id: str, poll_seconds: int = POLL_SECONDS) -> dict:
    while True:
        status = api_json("GET", f"/v1/jobs/{job_id}", timeout=15)
        print(f"{job_id}: {status['status']} | {status.get('stage', '')} | {status.get('message', '')}")
        if status["status"] in TERMINAL_STATUSES:
            if status["status"] == "FAILED":
                raise RuntimeError(status.get("error") or status.get("message") or "Job failed")
            return status
        time.sleep(poll_seconds)

def run_product(product: dict, feature_set: str) -> dict:
    job_id = submit_product(product, feature_set)
    wait_for_job(job_id)
    return api_json("GET", f"/v1/jobs/{job_id}/result", timeout=60)

def host_artifact_dir(result: dict) -> Path | None:
    row_id = (result.get("product") or {}).get("row_id")
    return PROJECT_ROOT / "data" / "artifacts" / row_id if row_id else None

def summarize_result(result: dict) -> dict:
    product = result.get("product") or {}
    search = result.get("search") or {}
    product_match = result.get("product_match") or {}
    acceptance = result.get("primary_url_acceptance") or {}
    evidence_set = result.get("evidence_set") or {}
    host_dir = host_artifact_dir(result)
    return {
        "row_id": product.get("row_id"),
        "job_status": result.get("job_status"),
        "coding_ready": result.get("coding_ready"),
        "primary_url": result.get("primary_url"),
        "supplementary_urls": result.get("supplementary_urls") or [],
        "serpapi_requests_used": search.get("serpapi_requests_used"),
        "search_stages": [stage.get("name") for stage in search.get("stages") or []],
        "selection_scope": product_match.get("selection_scope"),
        "url_decision_status": product_match.get("url_decision_status"),
        "strict_primary_url_accepted": acceptance.get("accepted"),
        "acceptance_reasons": acceptance.get("reasons") or [],
        "evidence_status": evidence_set.get("status"),
        "missing_features": evidence_set.get("missing_features") or [],
        "input_validation_warnings": product.get("input_validation_warnings") or [],
        "container_artifact_dir": result.get("artifact_dir"),
        "host_artifact_dir": str(host_dir) if host_dir else None,
    }

health = check_health()
print(f"Repository root: {PROJECT_ROOT}")
pprint(health)


## Single product

`main_text` and `country_code` are mandatory. `retailer_name`, `ean`, and `language_code` are optional. Supply EAN/GTIN as text so leading zeroes are preserved.


In [ ]:
FEATURE_SET = "example_features"

product = {
    "row_id": "ROW-001",
    "main_text": "Replace with the exact product identity text",
    "country_code": "CH",
    "retailer_name": None,
    "ean": None,
    "language_code": None,
}

result = run_product(product, FEATURE_SET)
pprint(summarize_result(result))


In [ ]:
# Inspect search stages and every final acceptance gate.
pprint(result.get("search") or {})
pprint(result.get("product_match") or {})
pprint(result.get("primary_url_acceptance") or {})
pprint(result.get("evidence_set") or {})
pprint(result.get("feature_assessments") or [])
pprint(result.get("browser_evidence") or [])

artifact_path = host_artifact_dir(result)
if artifact_path and artifact_path.is_dir():
    print(f"Host artifact directory: {artifact_path}")
    for name in ("review.md", "primary_url_acceptance.json"):
        path = artifact_path / name
        if path.is_file():
            print(f"\n--- {name} ---\n")
            print(path.read_text(encoding="utf-8"))
    candidates_path = artifact_path / "candidates.csv"
    if candidates_path.is_file():
        with candidates_path.open(newline="", encoding="utf-8-sig") as handle:
            candidate_rows = list(csv.DictReader(handle))
        print(f"Candidate rows: {len(candidate_rows)}")
        pprint(candidate_rows[:20])
else:
    print("No repository-local artifact directory was found for this result.")


## Optional CSV batch

Expected columns: `row_id, main_text, country_code, retailer_name, ean, language_code`. Batch summaries are written to `data/artifacts/notebook_batch_summary.csv`.


In [ ]:
BATCH_INPUT = PROJECT_ROOT / "inputs" / "products.csv"
BATCH_OUTPUT = PROJECT_ROOT / "data" / "artifacts" / "notebook_batch_summary.csv"

def blank_to_none(value: str | None) -> str | None:
    value = (value or "").strip()
    return value or None

def run_csv_batch(input_path: Path, output_path: Path, feature_set: str) -> list[dict]:
    if not input_path.is_file():
        raise FileNotFoundError(f"{input_path} does not exist")
    with input_path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    summaries = []
    for index, row in enumerate(rows, start=1):
        product = {
            "row_id": blank_to_none(row.get("row_id")) or f"ROW-{index:05d}",
            "main_text": blank_to_none(row.get("main_text")),
            "country_code": blank_to_none(row.get("country_code")),
            "retailer_name": blank_to_none(row.get("retailer_name")),
            "ean": blank_to_none(row.get("ean")),
            "language_code": blank_to_none(row.get("language_code")),
        }
        if not product["main_text"] or not product["country_code"]:
            raise ValueError(f"Row {index} is missing main_text or country_code")
        summary = summarize_result(run_product(product, feature_set))
        summaries.append({
            "row_id": summary["row_id"],
            "job_status": summary["job_status"],
            "coding_ready": summary["coding_ready"],
            "primary_url": summary["primary_url"],
            "serpapi_requests_used": summary["serpapi_requests_used"],
            "search_stages": "|".join(summary["search_stages"]),
            "selection_scope": summary["selection_scope"],
            "url_decision_status": summary["url_decision_status"],
            "strict_primary_url_accepted": summary["strict_primary_url_accepted"],
            "acceptance_reasons": "|".join(summary["acceptance_reasons"]),
            "missing_features": "|".join(summary["missing_features"]),
            "host_artifact_dir": summary["host_artifact_dir"],
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summaries[0]) if summaries else ["row_id"])
        writer.writeheader()
        writer.writerows(summaries)
    return summaries

# batch_results = run_csv_batch(BATCH_INPUT, BATCH_OUTPUT, FEATURE_SET)
# pprint(batch_results[:5])
